# Population disaggregation
Allocation of population data from coarser geographic areas (e.g. city-wide census) to finer spatial units (e.g. barangay, city blocks, buildings)

Two methods were used in this script to disaggregate population:
- based on land area
- based on building footprints

To check the effectiveness of the two methods, city-wide census data was disaggregated to barangay population estimates by:
- directly disaggregating city-wide data to barangay-level by proportion of land area **OR**
- disaggregating city-wide data to building-foot-print-level, then aggregating to barangay-level

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore", UserWarning)

In [2]:
folder = "ncr"
# folder = "rizal"

## Land-area-based population disaggregation
Allocated based on the proportion of each barangay's land area

In [3]:
city = gpd.read_file(f"..//data//{folder}//adm_bounds.gpkg", layer="city").to_crs(crs="EPSG:32651")
brgy = gpd.read_file(f"..//data//{folder}//adm_bounds.gpkg", layer="barangay").to_crs(crs="EPSG:32651")
# Removing areas that are not part of a barangay
if folder=="ncr":
    brgy = brgy.loc[~brgy["ADM4_EN"].isin(["Manila North Cemetery","Tutuban Mall (Claimed by Five Barangays of Tondo, Manila)"])].reset_index(drop=True)

In [4]:
brgy["area"] = brgy.area

brgy_m1 = brgy.merge(city[["ADM3_PCODE","pop"]],on="ADM3_PCODE",suffixes=(None,"_city"))

def get_pop_per_area(df):
    return df["area"]*(df["pop_city"].values[0]/df["area"].sum())    # barangay area x (city population/total city area)

brgy_m1["pop_est"] = brgy_m1.groupby("ADM3_PCODE",as_index=False).apply(get_pop_per_area)

brgy_m1.to_file(f"..//data//{folder}//brgy_pop_estimates.gpkg",driver="GPKG",layer="landarea_based")
brgy_m1[["ADM4_EN","ADM3_EN","ADM4_PCODE","pop","pop_est"]].to_csv(f"..//pop_estimates//{folder}_landarea_based.csv",index=False)

## Building-footprint-based population disaggregation
- Disaggregating city-wide data to each building footprint (allocated based on each building footprint's land area)
- Aggregated building footprint population estimate data to barangay-level

In [5]:
bldg = gpd.read_file(f"..//data//{folder}//buildings.gpkg", layer="bldg").to_crs(crs="EPSG:32651")

In [6]:
bldg["area"] = bldg.area
bldg['centroid'] = bldg.geometry.centroid
bldg_cent = bldg.set_geometry("centroid")

bldg_det = bldg_cent.sjoin(city[["ADM3_EN","ADM3_PCODE","geometry","pop"]],how="left",predicate="within")\
        .drop(columns=["index_right"])\
        .sjoin(brgy[["ADM4_EN","ADM4_PCODE","ADM3_EN","geometry"]],how="left",predicate="within")\
        .drop(columns=["index_right"])

def get_pop_per_area(df):
    return df["area"]*(df["pop"].values[0]/df["area"].sum())    # building footprint area x (city population/total city area)

bldg_det["pop_est"] = bldg_det.groupby("ADM3_PCODE",as_index=False).apply(get_pop_per_area)

# Aggregating to barangay-level
brgy_m2 = brgy.merge(bldg_det[["ADM4_PCODE","pop_est"]].groupby("ADM4_PCODE",as_index=False).sum(),how="left",on="ADM4_PCODE")

# Saving data
brgy_m2.to_file(f"..//data//{folder}//brgy_pop_estimates.gpkg",driver="GPKG",layer="building_based")
brgy_m2[["ADM4_EN","ADM3_EN","ADM4_PCODE","pop","pop_est"]].to_csv(f"..//pop_estimates//{folder}_building_based.csv",index=False)

In [7]:
# Saving building-level population estimate
bldg_det[["full_plus_code","pop_est"]].to_csv(f"..//data//{folder}//bldg_est_pop.csv",index=False)